# 10 — EDA Datos Físicos (SWaT)
Análisis exploratorio ligero sobre los sensores físicos ya limpios (post notebook 04b)
Objetivo: identificar qué sensores discriminan entre Normal y Ataque

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DELTA_NORMAL_PATH = "/Volumes/workspace/default/phisical_measures/delta_normal/"
DELTA_ATTACK_PATH = "/Volumes/workspace/default/phisical_measures/delta_attack/"

df_normal = spark.read.format("delta").load(DELTA_NORMAL_PATH)
df_attack  = spark.read.format("delta").load(DELTA_ATTACK_PATH)

df = df_normal.withColumn("label", F.lit(0)).union(
    df_attack.withColumn("label",
        F.when(F.col("Normal_Attack") == "Attack", 1).otherwise(0)
    )
).drop("Normal_Attack", "Timestamp")

total = df.count()
print(f"Total registros : {total:,}")
print(f"Columnas        : {len(df.columns)}")
df.printSchema()

## 1 — Distribución de clases

In [0]:
dist = (
    df.groupBy("label")
      .count()
      .withColumn("pct", F.round(F.col("count") / total * 100, 2))
      .orderBy("label")
      .toPandas()
)
print(dist.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors_cls = ["#4C8BF5", "#E8453C"]
axes[0].bar(["Normal (0)", "Ataque (1)"], dist["count"], color=colors_cls, edgecolor="white")
axes[0].set_title("Distribución de clases — Físicos", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Registros")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for i, row in dist.iterrows():
    axes[0].text(i, row["count"] + 1000, f"{row['pct']}%", ha="center", fontsize=11)
axes[1].pie(dist["count"], labels=["Normal (0)", "Ataque (1)"],
            colors=colors_cls, autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Proporción Normal / Ataque", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2 — Identificar columnas por tipo

In [0]:
exclude = ["timestamp_dt", "label"]

continuous_cols = [
    c for c in df.columns
    if c not in exclude
    and df.schema[c].dataType.simpleString() in ("double", "float")
]
discrete_cols = [
    c for c in df.columns
    if c not in exclude
    and df.schema[c].dataType.simpleString() in ("int", "integer", "long")
]

print(f"Sensores continuos  : {len(continuous_cols)}")
print(continuous_cols)
print(f"\nActuadores discretos: {len(discrete_cols)}")
print(discrete_cols)

## 3 — Correlación de cada sensor con el label

In [0]:
all_sensor_cols = continuous_cols + discrete_cols

pdf_corr = (
    df.select(all_sensor_cols + ["label"])
      .sample(fraction=0.2, seed=42)
      .toPandas()
)
for c in all_sensor_cols:
    pdf_corr[c] = pd.to_numeric(pdf_corr[c], errors="coerce")

corr_label = (
    pdf_corr[all_sensor_cols + ["label"]]
    .corr()["label"]
    .drop("label")
    .abs()
    .sort_values(ascending=False)
    .reset_index()
)
corr_label.columns = ["sensor", "corr_abs"]
print("Ranking completo de correlación con label:")
print(corr_label.to_string(index=False))

In [0]:
top20 = corr_label.head(20)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ["#E8453C" if v > 0.1 else "#4C8BF5" for v in top20["corr_abs"]]
ax.barh(top20["sensor"][::-1], top20["corr_abs"][::-1],
        color=bar_colors[::-1], edgecolor="white")
ax.axvline(0.1, color="gray", linestyle="--", lw=0.8, label="umbral 0.1")
ax.set_title("Top 20 sensores — correlación absoluta con label", fontsize=13, fontweight="bold")
ax.set_xlabel("Correlación absoluta")
ax.legend()
plt.tight_layout()
plt.show()

signal_cols = corr_label[corr_label["corr_abs"] >= 0.1]["sensor"].tolist()
noise_cols  = corr_label[corr_label["corr_abs"] <  0.1]["sensor"].tolist()
print(f"\nSensores con señal (corr >= 0.1) : {len(signal_cols)}")
print(signal_cols)
print(f"\nSensores sin señal (corr < 0.1)  : {len(noise_cols)}")
print(noise_cols)

## 4 — Distribución Normal vs Ataque para sensores con señal

In [0]:
pdf_signal = (
    df.select(signal_cols + ["label"])
      .sample(fraction=0.3, seed=42)
      .toPandas()
)
for c in signal_cols:
    pdf_signal[c] = pd.to_numeric(pdf_signal[c], errors="coerce")

pdf_n = pdf_signal[pdf_signal["label"] == 0]
pdf_a = pdf_signal[pdf_signal["label"] == 1]

n_cols_plot = 3
n_rows_plot = int(np.ceil(len(signal_cols) / n_cols_plot))
fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                         figsize=(5 * n_cols_plot, 3.5 * n_rows_plot))
axes_flat = np.array(axes).flatten()

for i, col in enumerate(signal_cols):
    ax = axes_flat[i]
    ax.hist(pdf_n[col].dropna(), bins=50, color="#4C8BF5", alpha=0.6, density=True, label="Normal")
    ax.hist(pdf_a[col].dropna(), bins=50, color="#E8453C", alpha=0.7, density=True, label="Ataque")
    corr_val = corr_label[corr_label["sensor"] == col]["corr_abs"].values[0]
    ax.set_title(f"{col}  (corr={corr_val:.3f})", fontsize=10, fontweight="bold")
    ax.set_ylabel("Densidad")
    ax.legend(fontsize=8)

for j in range(len(signal_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle("Distribución Normal vs Ataque — Sensores con señal",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5 — Actuadores discretos: distribución por clase

In [0]:
discrete_signal = [c for c in signal_cols if c in discrete_cols]

if discrete_signal:
    pdf_disc = df.select(discrete_signal + ["label"]).toPandas()
    for c in discrete_signal:
        pdf_disc[c] = pd.to_numeric(pdf_disc[c], errors="coerce")

    n_cols_d = min(3, len(discrete_signal))
    n_rows_d = int(np.ceil(len(discrete_signal) / n_cols_d))
    fig, axes = plt.subplots(n_rows_d, n_cols_d,
                             figsize=(5 * n_cols_d, 3.5 * n_rows_d))
    axes_d = np.array(axes).flatten()

    for i, col in enumerate(discrete_signal):
        ax = axes_d[i]
        vals = sorted(pdf_disc[col].dropna().unique())
        n_freq = pdf_disc[pdf_disc["label"] == 0][col].value_counts(normalize=True).reindex(vals, fill_value=0)
        a_freq = pdf_disc[pdf_disc["label"] == 1][col].value_counts(normalize=True).reindex(vals, fill_value=0)
        x = np.arange(len(vals))
        w = 0.35
        ax.bar(x - w/2, n_freq.values, w, color="#4C8BF5", alpha=0.85, label="Normal")
        ax.bar(x + w/2, a_freq.values, w, color="#E8453C", alpha=0.85, label="Ataque")
        ax.set_xticks(x)
        ax.set_xticklabels([str(int(v)) for v in vals])
        corr_val = corr_label[corr_label["sensor"] == col]["corr_abs"].values[0]
        ax.set_title(f"{col}  (corr={corr_val:.3f})", fontsize=10, fontweight="bold")
        ax.set_ylabel("Frecuencia relativa")
        ax.legend(fontsize=8)

    for j in range(len(discrete_signal), len(axes_d)):
        axes_d[j].set_visible(False)

    plt.suptitle("Actuadores discretos — frecuencia por estado y clase",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No hay actuadores discretos con señal suficiente (corr >= 0.1)")

## 6 — Estadísticas comparativas por clase

In [0]:
agg_exprs = [
    F.round(F.mean(F.col(c).cast("double")), 4).alias(f"mean_{c}")
    for c in signal_cols
]

stats = (
    df.groupBy("label")
      .agg(*agg_exprs)
      .orderBy("label")
      .toPandas()
)

stats_t = stats.set_index("label").T
stats_t.columns = ["Normal", "Ataque"]
stats_t["diferencia_%"] = (
    (stats_t["Ataque"] - stats_t["Normal"])
    / stats_t["Normal"].abs() * 100
).round(1)
stats_t = stats_t.sort_values("diferencia_%", key=abs, ascending=False)
stats_t.index = [c.replace("mean_", "") for c in stats_t.index]
print("Diferencia porcentual Normal -> Ataque por sensor:")
print(stats_t.to_string())

## 7 — Resumen EDA

In [0]:
print("=" * 55)
print("RESUMEN EDA — DATOS FÍSICOS")
print("=" * 55)
print(f"Total registros         : {total:,}")
dist_dict = dict(zip(dist["label"], dist["count"]))
pct_n = dist[dist["label"]==0]["pct"].values[0]
pct_a = dist[dist["label"]==1]["pct"].values[0]
print(f"  Normal (0)            : {dist_dict.get(0,0):,}  ({pct_n}%)")
print(f"  Ataque (1)            : {dist_dict.get(1,0):,}  ({pct_a}%)")
print(f"Sensores continuos      : {len(continuous_cols)}")
print(f"Actuadores discretos    : {len(discrete_cols)}")
print("-" * 55)
print(f"Con señal (corr >= 0.1) : {len(signal_cols)}")
print(f"  -> {signal_cols}")
print(f"Sin señal (ruido)       : {len(noise_cols)}")
print(f"  -> {noise_cols}")
print("-" * 55)
top_sensor = corr_label.iloc[0]
print(f"Top sensor              : {top_sensor['sensor']}  ({top_sensor['corr_abs']:.4f})")
print("=" * 55)